# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [1]:
from crewai import Agent, Task, Crew
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [3]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

### START CODE HERE ###

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance
scrape_website_tool = ScrapeWebsiteTool()

### END CODE HERE ###

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

In [4]:
# define the research planner agent
research_planner = Agent(
    role="Research Planner",
    goal="Analyze queries and break them down into smaller, specific research topics.",
    backstory=(
         "You are a research strategist who excels at breaking down complex questions "
         "into manageable research components. You identify what needs to be researched "
         "and create clear research objectives."
    ),
    verbose=True, # set to True to see detailed agent actions
    max_rpm=150,
    max_iter=15
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [5]:

researcher = Agent(
    role="Internet Researcher",
    goal="Research thoroughly all assigned topics",
    ### START CODE HERE ###
    backstory=(
        "You are a highly skilled internet researcher with expertise in finding accurate, "
        "relevant, and up-to-date information from diverse online sources. You use advanced "
        "search techniques and web scraping tools to gather comprehensive data for each task."
    ),
    # add the 2 tool instances you created
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify data for accuracy, identify inconsistencies, "
        "and flag potential misinformation"
    ),
    ### START CODE HERE ###
    backstory=(
        "You are a meticulous fact-checker with strong analytical skills. Your job is to "
        "validate information by cross-referencing multiple reliable sources, detecting "
        "inconsistencies, and ensuring all outputs are accurate and trustworthy."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

report_writer = Agent(
    role="Report Writer",
    goal="Write clear, concise, and well-structured reports based on gathered information",
    ### START CODE HERE ###
    backstory=(
        "You are a professional report writer who excels at synthesizing complex information "
        "into clear, structured, and engaging reports. You ensure logical flow, readability, "
        "and clarity while maintaining accuracy and coherence."
    ),
    ### END CODE HERE ###
    verbose=True,max_rpm=150,
    max_iter=15
)
        

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

In [6]:
# define the create research plan task
create_research_plan_task = Task(
    description=(
        "Based on the user's query, break it down into specific topics and key questions, "
        "and create a focused research plan."
        "The user's query is: {user_query}"
    ),
    expected_output=(
        "A research plan with main research topics to investigate, "
        "key questions for each topic, and success criteria for the research."
        ),
    agent=research_planner,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [7]:
# define the gather research data task
gather_research_data_task = Task(
    description=(
        "Using the research plan, collect information on all identified topics. "
        "Cite all sources used."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A detailed research document covering all topics, including summarized findings, "
        "key insights, and properly formatted source citations (URLs)."
    ),
    agent=researcher
    ### END CODE HERE ###
)

#define the verify information quality task
verify_information_quality_task = Task(
    description=(
        "Review all collected research. Identify any conflicting information, "
        "potential misinformation, or gaps that need addressing."
    ),
    ### START CODE HERE ###
     expected_output=(
        "A verified version of the research with corrected facts, flagged inconsistencies, "
        "confidence levels for key claims, and notes on any missing or uncertain information."
    ),
    agent=fact_checker
    ### END CODE HERE ###
)

# define the write final report task
write_final_report_task = Task(
    description=(
        "Create a comprehensive report that answers the original query using all verified research data. "
        "Structure it with clear sections, include citations, and provide actionable insights."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A well-structured final report with sections (introduction, analysis, findings, conclusion), "
        "clear explanations, and properly integrated citations."
    ),
    agent=report_writer
    ### END CODE HERE ###
)
    

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [8]:
# create the crew with the defined agents and tasks
crew = Crew(
    ### START CODE HERE ###
    agents=[research_planner, researcher, fact_checker, report_writer],
    tasks=[
        create_research_plan_task,
        gather_research_data_task,
        verify_information_quality_task,
        write_final_report_task
    ]
    ### 
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [9]:
### START CODE HERE ###

# Write your query, which will be used as input for the tasks.
user_query = "What are the latest trends in AI agents and their real-world applications?"

### END CODE HERE ###

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [10]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Based on the user's query, break it down into specific topics and key questions, and create a focused    │
│  research plan.The user's query is: What are the latest trends in AI agents and their real-world applications?  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan on Latest Trends in AI Agents and Their Real-World Applications                              │
│                                                                                                                 │
│  #### Main Research Topics to Investigate                                                                       │
│                                                                                                                 │
│  1. **Emerging AI Technologies and Models**                                                                     │
│     - **Key Questions:**                                                                                        │
│       - What are the latest advancements in AI algorithms (e.g., reinforcement learning, generative models)?    │
│       - How are transformer architectures evolving and being utilized in creating AI agents?                    │
│       - What role do pre-trained models and fine-tuning techniques play in enhancing AI capabilities?           │
│                                                                                                                 │
│  2. **Real-World Applications of AI Agents**                                                                    │
│     - **Key Questions:**                                                                                        │
│       - In which industries (e.g., healthcare, finance, retail) are AI agents currently being deployed?         │
│       - How are AI agents transforming customer service through chatbots and virtual assistants?                │
│       - What are some notable case studies showcasing the effectiveness of AI agents in real-world scenarios?   │
│                                                                                                                 │
│  3. **Ethical Considerations and Challenges**                                                                   │
│     - **Key Questions:**                                                                                        │
│       - What ethical dilemmas arise from the deployment of AI agents in various sectors?                        │
│       - How do biases in AI training data affect the outputs and decision-making of AI agents?                  │
│       - What are the implications of AI agents on employment and workforce dynamics?                            │
│                                                                                                                 │
│  4. **User Experience and Human-AI Interaction**                                                                │
│     - **Key Questions:**                                                                                        │
│       - How do users perceive and interact with AI agents in different contexts?                                │
│       - What are the best practices for designing user-friendly AI interfaces?                                  │
│       - How does the level of transparency in AI agent decisions impact user trust and acceptance?              │
│                                                                                                                 │
│  5. **Future Trends and Predictions**                                                                           │
│     - **Key Questions:**                               

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, collect information on all identified topics. Cite all sources used.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: Thought: I need to thoroughly research the latest trends in AI agents and their real-world            │
│  applications by exploring each topic systematically. I'll start with the first topic regarding emerging AI     │
│  technologies and models.                                                                                       │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "latest advancements in AI algorithms reinforcement learning generative models",             │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "best practices for designing user-friendly AI interfaces",                                  │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Design AI user experiences  |  web.dev                                                                  │
│  URL: https://web.dev/learn/ai/ux-patterns                                                                      │
│  ID: https://web.dev/learn/ai/ux-patterns                                                                       │
│  Score: None                                                                                                    │
│  Published Date: None                                                                                           │
│  Author: None                                                                                                   │
│  Image: https://web.dev/./card.png                                                                              │
│  Favicon:                                                                                                       │
│  https://www.gstatic.com/devrel-devsite/prod/v579073a50c63499824df5a68b8922367066583d283ef78fdade1028efdb4ceb5  │
│  /web/images/favicon.png                                                                                        │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: # Design AI user experiences                                                                             │
│                                                                                                                 │
│  From a user's perspective, AI systems are inherently uncertain. They can produce                               │
│  inconsistent results and make regular mistakes. The user interface offers many                                 │
│  opportunities to absorb, filter, and reduce frustrations caused by AI                                          │
│  limitations. As a developer, you play a central role in shaping AI user                                        │
│  experiences because you have deeper insight into how and where an AI system                                    │
│  is likely to fail.                                                                                             │
│                                                                                                                 │
│  One key design consideration is how much control the users have over AI.                                       │
│  Some opportunities are invisible to users, while others                                                        │
│  have explicit interaction. Greater exposure means more flexibility, but also                                   │
│  more risk and complexity that have to be managed.                                                              │
│                                                                                                                 │
│  In this module, we learn best practices for designing user experience (UX)                                     │
│  patterns for three types of exposure: background, constrained, and open-ended.                                 │
│  For each type, we highlight how technical and architectural choices affect the                                 │
│  AI system's user experience.                                                                                   │
│                                                                                                                 │
│  ## Background AI                                                                                               │
│                                                       

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Verified Research on Latest Trends in AI Agents and Their Real-World Applications                          │
│                                                                                                                 │
│  #### Emerging AI Technologies and Models                                                                       │
│  1. **Latest Advancements in AI Algorithms**:                                                                   │
│     - **Claim**: Advancements in reinforcement learning (RL) and generative models, with HiPER improving        │
│  hierarchical training for LLMs.                                                                                │
│       - **Source Validity**: Verified through multiple sources, including [HiPER - Hierarchical Reinforcement   │
│  Learning](https://www.arxiv.org/abs/2602.16165) (Confidence Level: High).                                      │
│                                                                                                                 │
│  2. **Evolution of Transformer Architectures**:                                                                 │
│     - **Claim**: Advancements in transformers focus on pre-trained models and fine-tuning for performance       │
│  optimization.                                                                                                  │
│       - **Source Validity**: Confirmed through extensive research including [Reinforcement Learning for         │
│  Generative AI](https://arxiv.org/html/2308.00031v3) (Confidence Level: High).                                  │
│                                                                                                                 │
│  3. **Role of Pre-trained Models**:                                                                             │
│     - **Claim**: Generative AI's effectiveness hinges on pre-trained models.                                    │
│       - **Source Validity**: Supported by multiple peer-reviewed articles (Confidence Level: High).             │
│                                                                                                                 │
│  #### Real-World Applications of AI Agents                                                                      │
│  1. **Healthcare**:                                                                                             │
│     - **Claim**: AI agents automate prior authorizations, as demonstrated by Nirmitee's agent reducing          │
│  workflow time from 14 days to under 3 hours for 85% of requests.                                               │
│       - **Source Validity**: Confirmed through [Nirmitee Case                                                   │
│  Study](https://nirmitee.io/case-study/ai-agents-eliminating-prior-authorization-bottleneck-healthcare/)        │
│  (Confidence Level: High).                                                                                      │
│                                                                                                                 │
│  2. **Finance**:                                                                                                │
│     - **Claim**: AI agents enhance fraud detection and investment recommendations.                              │
│       - **Source Validity**: Details are provided but n

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query using all verified research data.          │
│  Structure it with clear sections, include citations, and provide actionable insights.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Research Document: Latest Trends in AI Agents and Their Real-world Applications                              │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│  Artificial Intelligence (AI) agents have become an integral part of various industries, transforming           │
│  operations and enhancing user experiences. This report presents a thorough analysis of the latest trends in    │
│  AI technologies and their real-world applications. The insights gathered aim to address critical questions     │
│  concerning advancements in AI algorithms, practical uses of AI agents across several sectors, ethical          │
│  challenges, user interactions, and future predictions for AI development.                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Emerging AI Technologies and Models                                                                         │
│                                                                                                                 │
│  ### Latest Advancements in AI Algorithms                                                                       │
│  AI advancements, particularly in reinforcement learning (RL) and generative models, mark a new era for         │
│  creating highly efficient and adaptive AI agents. For instance, hierarchical reinforcement learning            │
│  frameworks such as HiPER have improved the training patterns for large language models (LLMs), thereby         │
│  enhancing performance in multi-turn decision-making tasks and optimizing long-horizon task execution (Source:  │
│  [HiPER - Hierarchical Reinforcement Learning](https://www.arxiv.org/abs/2602.16165)).                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Evolution of Transformer Architectures                                                                     │
│  The evolution of transformer architectures continues to be pivotal in generative AI, with a focus on           │
│  pre-trained models that enhance AI capabilities. These models leverage fine-tuning techniques that improve     │
│  their adaptability across multiple applications without exhaustive retraining (Source: [Reinforcement          │
│  Learning for Generative AI](https://arxiv.org/html/2308.00031v3)).                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                        

From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [11]:
from IPython.display import Markdown
Markdown(result.raw) 

# Research Document: Latest Trends in AI Agents and Their Real-world Applications

## Introduction
Artificial Intelligence (AI) agents have become an integral part of various industries, transforming operations and enhancing user experiences. This report presents a thorough analysis of the latest trends in AI technologies and their real-world applications. The insights gathered aim to address critical questions concerning advancements in AI algorithms, practical uses of AI agents across several sectors, ethical challenges, user interactions, and future predictions for AI development.

---

## Emerging AI Technologies and Models

### Latest Advancements in AI Algorithms
AI advancements, particularly in reinforcement learning (RL) and generative models, mark a new era for creating highly efficient and adaptive AI agents. For instance, hierarchical reinforcement learning frameworks such as HiPER have improved the training patterns for large language models (LLMs), thereby enhancing performance in multi-turn decision-making tasks and optimizing long-horizon task execution (Source: [HiPER - Hierarchical Reinforcement Learning](https://www.arxiv.org/abs/2602.16165)).

---

### Evolution of Transformer Architectures
The evolution of transformer architectures continues to be pivotal in generative AI, with a focus on pre-trained models that enhance AI capabilities. These models leverage fine-tuning techniques that improve their adaptability across multiple applications without exhaustive retraining (Source: [Reinforcement Learning for Generative AI](https://arxiv.org/html/2308.00031v3)).

---

### Role of Pre-trained Models
Pre-trained models play a significant role in AI effectiveness as they are scalable and capable of performing diverse tasks. These models exemplify efficiency since they do not require extensive training on every new dataset (Source: [Generative AI advancements](https://arxiv.org/pdf/2604.10701)).

---

## Real-world Applications of AI Agents

### Healthcare
AI agents revolutionize healthcare through automation of various processes, particularly in streamlining prior authorization workflows. Nirmitee's AI solution enhances efficiency by completing 85% of these tasks, reducing processing time from 14 days to less than three hours, thereby significantly improving patient care (Source: [Nirmitee case study](https://nirmitee.io/case-study/ai-agents-eliminating-prior-authorization-bottleneck-healthcare/)). Additionally, virtual assistants are employed for patient intake and administrative functions like appointment scheduling and insurance claims (Source: [AI Agents in Healthcare](https://eminentcoders.com/ai-agents-in-healthcare-finance-and-retail/)).

---

### Finance
In the finance sector, AI agents excel in fraud detection and personalized investment recommendations, expediting decision-making processes involved in lending. Furthermore, they facilitate the automation of documentation tasks, making loan approvals more efficient (Source: [AI Agents in Finance](https://eminentcoders.com/ai-agents-in-healthcare-finance-and-retail/)).

---

### Retail
AI agents, used by retailers such as Klarna, enhance customer service through personalized recommendations and faster response times. Klarna's AI assistant reportedly manages workload equivalent to 700 full-time agents, demonstrating significant operational efficiencies (Source: [Klarna Case Study](https://openai.com/customer-stories/klarna/)).

---

## Ethical Considerations and Challenges

### Ethical Dilemmas
The deployment of AI agents raises ethical discussions surrounding autonomy, transparency, and accountability. It is crucial to consider frameworks like the EU AI Act, which advocates for ethical AI deployment while addressing concerns of efficiency over ethical implications (Source: [Systemic Risks of Agentic AI](https://www.acm.org/binaries/content/assets/public-policy/europe-tpc/systemic_risks_agentic_ai_policy-brief_final.pdf)).

---

### Bias in AI Training Data
Biases embedded in training data can significantly impact AI decision-making, leading to discriminatory outcomes. Addressing these biases is essential to uphold ethical standards in AI practices (Source: [Ethical Perspectives](https://link.springer.com/article/10.1007/s43681-025-00920-4)).

---

### Implications for Employment
As automation permeates various sectors, job displacement raises concerns about the labor market impacts and the pressing necessity for workforce reskilling initiatives (Source: [We need a new ethics for a world of AI agents](http://www.nature.com/articles/s41586-025-10021-1)).

---

## User Experience and Human-AI Interaction

### Designing Human-Centric Interactions
The design of AI interfaces should prioritize user-centric principles, which include creating transparency and fostering trust. Best practices involve using **Intent Previews** and **Autonomy Dials** to enhance user comprehension of AI actions (Source: [Designing AI agents that know when to step back](https://www.amazon.science/blog/designing-ai-agents-that-know-when-to-step-back)).

---

### Visibility and Feedback
Ensuring users understand AI agent decisions and providing consistent feedback can enhance user experience. Effective communication of status and transparency during interactions fosters trust and collaborative engagement with AI systems (Source: [Human-AI Collaboration](https://cltc.berkeley.edu/publication/ux-design-considerations-for-human-ai-agent-interaction/)).

---

### User Preferences
AI agents must continuously adapt to user preferences to cultivate lasting trust and improve user interaction over time (Source: [UX design for agents - Microsoft Design](https://microsoft.design/articles/ux-design-for-agents)).

---

## Future Trends and Predictions

### Autonomy and Multi-Agent Systems
By 2026, a considerable segment of enterprise applications is expected to adopt autonomous AI agents capable of sophisticated decision-making, significantly impacting business functionality (Source: [Google Cloud Trends](https://cloud.google.com/resources/content/ai-agent-trends-2026)).

---

### Shift Toward Personalization
The landscape of customer interaction will shift towards hyper-personalization, as AI agents evolve to provide more intuitive and customized services (Source: [Predictive Markets](https://blog.google/products/google-cloud/ai-business-trends-report-2026)).

---

### Increasing Regulation
As AI technology advances, the regulatory environment is poised for maturation, implementing stricter compliance measures that address transparency, accountability, and ethical considerations (Source: [AI Predictions](https://my.idc.com/getdoc.jsp?containerId=US53860925)).

---

## Conclusion
The advancements in AI agents signify a transformative shift across industries, enhancing operational efficiencies and user experiences. Despite the immense potential, ethical complexities and regulatory considerations necessitate vigilance and thoughtful governance. Proactive engagement with these challenges is vital as AI technologies proliferate and evolve in everyday applications.

---

### Source Citations
- [HiPER: Hierarchical Reinforcement Learning](https://www.arxiv.org/abs/2602.16165)
- [Reinforcement Learning for Generative AI](https://arxiv.org/html/2308.00031v3)
- [Nirmitee case study](https://nirmitee.io/case-study/ai-agents-eliminating-prior-authorization-bottleneck-healthcare/)
- [Klarna Case Study](https://openai.com/customer-stories/klarna/)
- [Systemic Risks of Agentic AI](https://www.acm.org/binaries/content/assets/public-policy/europe-tpc/systemic_risks_agentic_ai_policy-brief_final.pdf)
- [Ethical Perspectives](https://link.springer.com/article/10.1007/s43681-025-00920-4)
- [We need a new ethics for a world of AI agents](http://www.nature.com/articles/s41586-025-10021-1)
- [Designing AI agents that know when to step back](https://www.amazon.science/blog/designing-ai-agents-that-know-when-to-step-back)
- [Human-AI Collaboration](https://cltc.berkeley.edu/publication/ux-design-considerations-for-human-ai-agent-interaction/)
- [Google Cloud Trends](https://cloud.google.com/resources/content/ai-agent-trends-2026)
- [AI Predictions](https://my.idc.com/getdoc.jsp?containerId=US53860925)

This report encapsulates a comprehensive examination of AI agents' trends, applications, ethical dimensions, user perspectives, and future outlook, ensuring all information is well-organized and easily accessible.

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!